In [1]:
import sys
sys.path.append("../src")
from utils import *

class ControlSignals:
    def __init__(self):
        self.edge_flipflop1 = EdgeTriggeredDTypeFlipFlopWithPresetAndClear()
        self.edge_flipflop2 = EdgeTriggeredDTypeFlipFlopWithPresetAndClear()
        self.invert1 = Inverter()
        self.and_gate = AND()
    def __call__(self, clock):
        not_clk = self.invert1([clock])

        # while True:
        #     old_q1, old_q_bar1 = self.edge_flipflop1.getQ(), self.edge_flipflop1.getQ_bar()
        #     old_q2, old_q_bar2 = self.edge_flipflop2.getQ(), self.edge_flipflop2.getQ_bar()

        self.edge_flipflop1([self.edge_flipflop1.getQ_bar(), clock])
        self.edge_flipflop2([self.edge_flipflop1.getQ(), not_clk])
        clk_input_to_counter = self.edge_flipflop1.getQ()
        pulse = self.and_gate([self.edge_flipflop1.getQ_bar(), self.edge_flipflop2.getQ()])
        return [clk_input_to_counter, pulse]




In [27]:
def test_control_signals():
    """Tests the control signals for the RAM_8_8, ensuring correct timing and sequencing of read/write operations."""
    control_signals = ControlSignals()
    my_oscillator = Oscillator()
    for _ in range(20):
        clock = my_oscillator.level()
        clk_input_to_counter, pulse = control_signals(clock)
        # clk_input_to_counter, pulse = output[0], output[1]
        print(f"Clock: {clock}, Counter Clock Input: {clk_input_to_counter}, Pulse: {pulse}")
        my_oscillator.tick()

test_control_signals()

Clock: 0, Counter Clock Input: 0, Pulse: 0
Clock: 1, Counter Clock Input: 1, Pulse: 0
Clock: 0, Counter Clock Input: 1, Pulse: 0
Clock: 1, Counter Clock Input: 0, Pulse: 1
Clock: 0, Counter Clock Input: 0, Pulse: 0
Clock: 1, Counter Clock Input: 1, Pulse: 0
Clock: 0, Counter Clock Input: 1, Pulse: 0
Clock: 1, Counter Clock Input: 0, Pulse: 1
Clock: 0, Counter Clock Input: 0, Pulse: 0
Clock: 1, Counter Clock Input: 1, Pulse: 0
Clock: 0, Counter Clock Input: 1, Pulse: 0
Clock: 1, Counter Clock Input: 0, Pulse: 1
Clock: 0, Counter Clock Input: 0, Pulse: 0
Clock: 1, Counter Clock Input: 1, Pulse: 0
Clock: 0, Counter Clock Input: 1, Pulse: 0
Clock: 1, Counter Clock Input: 0, Pulse: 1
Clock: 0, Counter Clock Input: 0, Pulse: 0
Clock: 1, Counter Clock Input: 1, Pulse: 0
Clock: 0, Counter Clock Input: 1, Pulse: 0
Clock: 1, Counter Clock Input: 0, Pulse: 1


In [2]:
class Counter16Bit:
    """A Ripple Counter that counts"""
    def __init__(self,):
        self.nbits = 16
        self.flipflops = [EdgeTriggeredDTypeFlipFlopWithPresetAndClear() for _ in range(self.nbits)]
        self.and_gate = AND()

    def getQs(self):
        qs = [ff.getQ() for ff in self.flipflops]
        return list(reversed(qs))
    
    def SetMaxAddr(self):
        for ff in self.flipflops:
            ff([0, 0, 1, 0]) # D = 0, CLK = 0, PRE = 1, CLR = 0

    def __call__(self, clk = 0, clear_wire = 0):
        # The hardware stabilization loop
        while True:
            # 1. Take a snapshot of the current state of the ff
            old_qs = [ff.getQ() for ff in self.flipflops]

            # 2. Wire up the Async Clear
            # ff[0] is Q0 (LSB), ff[1] is Q1, ff[2] is Q2, ff[3] is Q3 (MSB)
            # We want to clear when we hit 10 (Binary: 1010). 
            # This happens when Q3 == 1 and Q1 == 1.
            # q1 = self.flipflops[1].getQ()
            # q3 = self.flipflops[3].getQ()
            # clear_wire = self.and_gate([q3, q1])

            # 3. Propagate the clock and data through the ripple chain
            current_clk = clk
            for ff in self.flipflops:
                data = ff.getQ_bar()
                # Pass Data, Clock, Preset(0), and clear wire (0)
                q, q_bar = ff([data, current_clk, 0, clear_wire])
                current_clk = q_bar

            # 4. Read the new state
            new_qs = [ff.getQ() for ff in self.flipflops]

            # 5. If the electrons have settled and nothing changed, break!
            if old_qs == new_qs:
                break

        # Return with MSB on the left for easy reading
        return list(reversed(new_qs))

In [29]:
def test_counter16Bit():
    my_oscillator = Oscillator()
    nbits = 16
    my_counter = Counter16Bit()
    
    previous_clock_state = -1
    golden_clock_state = 0
    golden_qs = [0] * nbits
    golden_value = 0
    for clk in range(65537):
        assert golden_clock_state == my_oscillator.level(), f"At clock {clk}, expected clock state {golden_clock_state} but got {my_oscillator.level()}"
        my_counter(my_oscillator.level())
        if previous_clock_state == 0 and golden_clock_state == 1:  # Only increment the expected counter value on the rising edge of the clock
            golden_value = (golden_value + 1) % (2 ** nbits)  # Increment and wrap around at 2^nbits
            golden_qs = int_to_nbit_list(golden_value, nbits)

        assert golden_qs == my_counter.getQs(), f"At clock {clk}, expected counter value {golden_value} but got {int(''.join(str(bit) for bit in my_counter.getQs()))}"
        previous_clock_state = golden_clock_state
        my_oscillator.tick()  # Move to the next clock state for the next iteration
        golden_clock_state = 1 - golden_clock_state  # Toggle the expected clock state for

test_counter16Bit()

In [30]:
def test_counter16BitMaxAddr():
    nbits = 16
    my_counter = Counter16Bit()
    
    my_counter.SetMaxAddr()
    golden_value = 65535
    golden_qs = int_to_nbit_list(golden_value, nbits)
    assert golden_qs == my_counter.getQs(), f"expected counter value {golden_value} but got {int(''.join(str(bit) for bit in my_counter.getQs()))}"
    
test_counter16BitMaxAddr()

In [31]:
class AutomatedAccumulatingAdder:
    """
    A class representing an automated accumulating adder.
    """
    def __init__(self, nbits = 8):
        self.nbits = nbits
        self.counter = Counter16Bit()  # Using the 16-bit counter as an accumulator
        # self.control_signals = ControlSignals()
        self.ram64KB = RAM_64KB()

        # NBitsAccumulator
        self.adder = NBitAdderWithOverflow(self.nbits)
        self.register = NBitsEdgeTriggeredDTypeFlipFlopWithPresetAndClear(self.nbits)

        self.nor_gate = NOR(self.nbits) # for RAM Write signal
        self.and_gate = AND(2)

    def SetMaxAddr(self):
        self.counter.SetMaxAddr()

    def __call__(self, clk_input_to_counter, pulse):
        # clk_input_to_counter, pulse = self.control_signals(clock)
        self.counter(clk_input_to_counter)
        addr = self.counter.getQs()
        data_out_from_memory = self.ram64KB.read(addr)
        overflow, sum_bits = self.adder(data_out_from_memory, self.register.getQ())
        self.register(sum_bits, pulse)  # Data, Clock, Preset(0), Clear(0)

        # from page 297 in book, author
        # wants to write the accumulated sum to the first memory
        # location that has a value of 00h.
        ram_write = self.and_gate([self.nor_gate(data_out_from_memory), pulse])
        if ram_write:
            self.write_to_memory(addr, self.register.getQ())

    def write_to_memory(self, address, data):
        self.ram64KB(address, data, write=1)

    def read_register(self):
        return self.register.getQ()        

In [32]:

nbits = 8
control_signals = ControlSignals()
my_oscillator = Oscillator()
my_accumulator_adder = AutomatedAccumulatingAdder(nbits)
my_accumulator_adder.SetMaxAddr()

# 1. write to memory 
addrs = [[0,0,0,0, 0,0,0,0, 0,0,0,0, 0,0,0,0],
         [0,0,0,0, 0,0,0,0, 0,0,0,0, 0,0,0,1],
         #[0,0,0,0, 0,0,0,0, 0,0,0,0, 0,0,1,0]
        # [0,0,0,0, 0,0,0,0, 0,0,0,0, 0,0,1,1]
        # [0,0,0,0, 0,0,0,0, 0,0,0,0, 0,1,0,0]
        # [0,0,0,0, 0,0,0,0, 0,0,0,0, 0,1,0,1]
        # [0,0,0,0, 0,0,0,0, 0,0,0,0, 0,1,1,0]
        # [0,0,0,0, 0,0,0,0, 0,0,0,0, 0,1,1,1]
        ]
datas = [53, 27, #9, 49, 30, 18, 35, 12
         
        ]

dst_addr = [0,0,0,0, 0,0,0,0, 0,0,0,0, 0,0,1,0]
for addr, data in zip(addrs, datas):
    my_accumulator_adder.write_to_memory(addr,  int_to_nbit_list(data, nbits))

# old print
print("Initial register value:", my_accumulator_adder.ram64KB.read(addrs[0]))
print("Initial register value:", my_accumulator_adder.ram64KB.read(addrs[1]))
for idx in range(20):
    clock = my_oscillator.level()
    clk_input_to_counter, pulse = control_signals(clock)
    operand_B = my_accumulator_adder.read_register() 
    my_accumulator_adder(clk_input_to_counter, pulse)
    operand_A = my_accumulator_adder.ram64KB.read(my_accumulator_adder.counter.getQs())
    register = my_accumulator_adder.read_register()

    print(f"idx: {idx}, Clock: {clock}, Counter Clock: {clk_input_to_counter}, Pulse: {pulse}, A: {operand_A} + B: {operand_B}, register: {register}, dst_addr_val: {my_accumulator_adder.ram64KB.read(dst_addr)}")
    my_oscillator.tick()


Initial register value: [0, 0, 1, 1, 0, 1, 0, 1]
Initial register value: [0, 0, 0, 1, 1, 0, 1, 1]
idx: 0, Clock: 0, Counter Clock: 0, Pulse: 0, A: [0, 0, 0, 0, 0, 0, 0, 0] + B: [0, 0, 0, 0, 0, 0, 0, 0], register: [0, 0, 0, 0, 0, 0, 0, 0], dst_addr_val: [0, 0, 0, 0, 0, 0, 0, 0]
idx: 1, Clock: 1, Counter Clock: 1, Pulse: 0, A: [0, 0, 1, 1, 0, 1, 0, 1] + B: [0, 0, 0, 0, 0, 0, 0, 0], register: [0, 0, 0, 0, 0, 0, 0, 0], dst_addr_val: [0, 0, 0, 0, 0, 0, 0, 0]
idx: 2, Clock: 0, Counter Clock: 1, Pulse: 0, A: [0, 0, 1, 1, 0, 1, 0, 1] + B: [0, 0, 0, 0, 0, 0, 0, 0], register: [0, 0, 0, 0, 0, 0, 0, 0], dst_addr_val: [0, 0, 0, 0, 0, 0, 0, 0]
idx: 3, Clock: 1, Counter Clock: 0, Pulse: 1, A: [0, 0, 1, 1, 0, 1, 0, 1] + B: [0, 0, 0, 0, 0, 0, 0, 0], register: [0, 0, 1, 1, 0, 1, 0, 1], dst_addr_val: [0, 0, 0, 0, 0, 0, 0, 0]
idx: 4, Clock: 0, Counter Clock: 0, Pulse: 0, A: [0, 0, 1, 1, 0, 1, 0, 1] + B: [0, 0, 1, 1, 0, 1, 0, 1], register: [0, 0, 1, 1, 0, 1, 0, 1], dst_addr_val: [0, 0, 0, 0, 0, 0, 0, 0]
id

In [11]:

nbits = 8
control_signals = ControlSignals()
my_oscillator = Oscillator()
my_accumulator_adder = AutomatedAccumulatingAdder(nbits)

# 1. write to memory 
addrs = [#[0,0,0,0, 0,0,0,0, 0,0,0,0, 0,0,0,0],
         [0,0,0,0, 0,0,0,0, 0,0,0,0, 0,0,0,1],
         [0,0,0,0, 0,0,0,0, 0,0,0,0, 0,0,1,0]
        # [0,0,0,0, 0,0,0,0, 0,0,0,0, 0,0,1,1]
        # [0,0,0,0, 0,0,0,0, 0,0,0,0, 0,1,0,0]
        # [0,0,0,0, 0,0,0,0, 0,0,0,0, 0,1,0,1]
        # [0,0,0,0, 0,0,0,0, 0,0,0,0, 0,1,1,0]
        # [0,0,0,0, 0,0,0,0, 0,0,0,0, 0,1,1,1]
        ]
datas = [53, 27, #9, 49, 30, 18, 35, 12
         
        ]

dst_addr = [0,0,0,0, 0,0,0,0, 0,0,0,0, 0,0,1,1]
for addr, data in zip(addrs, datas):
    my_accumulator_adder.write_to_memory(addr,  int_to_nbit_list(data, nbits))

print("Initial register value:", my_accumulator_adder.ram64KB.read(addrs[0]))
print("Initial register value:", my_accumulator_adder.ram64KB.read(addrs[1]))
for idx in range(20):
    clock = my_oscillator.level()
    clk_input_to_counter, pulse = control_signals(clock)
    operand_B = my_accumulator_adder.read_register() 
    my_accumulator_adder(clk_input_to_counter, pulse)
    operand_A = my_accumulator_adder.ram64KB.read(my_accumulator_adder.counter.getQs())
    register = my_accumulator_adder.read_register()

    print(f"idx: {idx}, Clock: {clock}, Counter Clock: {clk_input_to_counter}, Pulse: {pulse}, A: {operand_A} + B: {operand_B}, register: {register}, dst_addr_val: {my_accumulator_adder.ram64KB.read(dst_addr)}")
    my_oscillator.tick()


Initial register value: [0, 0, 1, 1, 0, 1, 0, 1]
Initial register value: [0, 0, 0, 1, 1, 0, 1, 1]
idx: 0, Clock: 0, Counter Clock: 0, Pulse: 0, A: [0, 0, 0, 0, 0, 0, 0, 0] + B: [0, 0, 0, 0, 0, 0, 0, 0], register: [0, 0, 0, 0, 0, 0, 0, 0], dst_addr_val: [0, 0, 0, 0, 0, 0, 0, 0]
idx: 1, Clock: 1, Counter Clock: 1, Pulse: 0, A: [0, 0, 1, 1, 0, 1, 0, 1] + B: [0, 0, 0, 0, 0, 0, 0, 0], register: [0, 0, 0, 0, 0, 0, 0, 0], dst_addr_val: [0, 0, 0, 0, 0, 0, 0, 0]
idx: 2, Clock: 0, Counter Clock: 1, Pulse: 0, A: [0, 0, 1, 1, 0, 1, 0, 1] + B: [0, 0, 0, 0, 0, 0, 0, 0], register: [0, 0, 0, 0, 0, 0, 0, 0], dst_addr_val: [0, 0, 0, 0, 0, 0, 0, 0]
idx: 3, Clock: 1, Counter Clock: 0, Pulse: 1, A: [0, 0, 1, 1, 0, 1, 0, 1] + B: [0, 0, 0, 0, 0, 0, 0, 0], register: [0, 0, 1, 1, 0, 1, 0, 1], dst_addr_val: [0, 0, 0, 0, 0, 0, 0, 0]
idx: 4, Clock: 0, Counter Clock: 0, Pulse: 0, A: [0, 0, 1, 1, 0, 1, 0, 1] + B: [0, 0, 1, 1, 0, 1, 0, 1], register: [0, 0, 1, 1, 0, 1, 0, 1], dst_addr_val: [0, 0, 0, 0, 0, 0, 0, 0]
id

In [3]:

class Decoder_2_4:
    """
    2 to 4 decoder, takes 2 input bits and decodes them into 4 output lines.
    Only one output line will be high (1) at a time
    for example, if the input is 01, then the output will be 0010
    """
    def __init__(self, nin = 2, nout = 4):
        self.nin = nin
        self.nout = nout
        self.inverters = [Inverter() for _ in range(self.nin)]
        self.and_gates = [AND(2) for _ in range(self.nout)]
        self.and_gates_with_write = [AND(2) for _ in range(self.nout)]
    
    def write(self, inputs, write):
        assert len(inputs) == self.nout, f"Inputs must be {self.nout} bits long"
        assert write in [0, 1], "Write signal must be 0 or 1"
        # This function will take the write signal and the inputs, and return the output of the AND gate
        outputs = []
        for i in range(self.nout):
            outputs.append(self.and_gates_with_write[i]([write, inputs[i]]))
        return outputs

    def __call__(self, address):
        assert len(address) == self.nin, "Input must be 2 bits long"
        idxs = [
            [0, 0], 
            [0, 1], 
            [1, 0], 
            [1, 1], 
        ]
        address = [[address[i]] for i in range(len(address))] # convert to list of lists for inverters gate input
        output = [0] * self.nout
        for i, idx in enumerate(idxs):
            input_and = []
            for j, val in enumerate(idx):
                if val == 0:
                    input_and.append(self.inverters[j](address[j]))
                else:
                    input_and.append(address[j][0])
            output[self.nout - 1 - i] = self.and_gates[i](input_and)
        # output[0] is MSB
        return output

In [4]:
decoder_2_4 = Decoder_2_4()
nbits = 4
inputs = [[0, 0], [0, 1], [1, 0], [1, 1]]


for idx, input in enumerate(inputs):
    output = decoder_2_4(input)
    print(f"input: {input}, output: {output}")


input: [0, 0], output: [0, 0, 0, 1]
input: [0, 1], output: [0, 0, 1, 0]
input: [1, 0], output: [0, 1, 0, 0]
input: [1, 1], output: [1, 0, 0, 0]


In [7]:
class AutomatedAccumulatingAdderV2:
    """
    A class representing an automated accumulating adder.
    """
    def __init__(self, nbits = 8):
        self.nbits = nbits
        self.counter = Counter16Bit()  # Using the 16-bit counter as an accumulator
        # self.control_signals = ControlSignals()
        self.ram64KB = RAM_64KB()

        # NBitsAccumulator
        self.adder = NBitAdderWithOverflow(self.nbits)

        self.inst_latch = NBitsEdgeTriggeredDTypeFlipFlopWithPresetAndClear(self.nbits)

        self.low_latch = NBitsEdgeTriggeredDTypeFlipFlopWithPresetAndClear(self.nbits+1)    # overflow of low
        self.middle_latch = NBitsEdgeTriggeredDTypeFlipFlopWithPresetAndClear(self.nbits+1) # overflow of middle
        self.high_latch = NBitsEdgeTriggeredDTypeFlipFlopWithPresetAndClear(self.nbits)   

        self.decoder_2_4_for_clock = Decoder_2_4()
        self.and_gate_use_for_instruct_clk = AND()
        self.and_gate_use_for_low_byte_clk = AND(3)
        self.and_gate_use_for_middle_byte_clk = AND(3)
        self.and_gate_use_for_high_byte_clk = AND(3)

        self.xor_gates_for_complements = [XOR() for _ in range(self.nbits)]
        
        self.and_gate_for_carry_in_adder = AND()
        self.or_gate_for_carry_in_adder = OR()

        self.invert_for_write_ram = Inverter()
        self.and_gate_for_write_ram = AND(3)

        self.enable_low_byte = -1
        self.enable_mid_byte = -1
        self.enable_hig_byte = -1

        self.carry_in_adder = -1
        self.input_a = [-1] * self.nbits

        self.enable_instruct = -1

        self.halt_instruct = -1
        # self.invert_gate_for_halt = Inverter()


    def SetMaxAddr(self):
        self.counter.SetMaxAddr()

    def getEnableLow(self):
        return self.enable_low_byte
    
    def getEnableMiddle(self):
        return self.enable_mid_byte
    
    def getEnableHigh(self):
        return self.enable_hig_byte
    
    def getHaltInst(self):
        return self.halt_instruct
    
    def getCarryInAdder(self):
        return self.carry_in_adder

    def getInputA(self):
        return self.input_a
    
    def getInputA(self):
        return self.input_a

    def getEnableInstruc(self):
        return self.enable_instruct

    def __call__(self, clk_input_to_counter, pulse):
        # clk_input_to_counter, pulse = self.control_signals(clock)
        self.counter(clk_input_to_counter)
        addr = self.counter.getQs()
        # instruct code or data
        data_out_from_memory = self.ram64KB.read(addr)

        # page 305: The three enable signals for the tri-state buffers can be 
        # generated by a 2-to-4 decoder using least significant bits of the memory address.
        enable_high_byte, enable_middle_byte, enable_low_byte, enable_insruct = self.decoder_2_4_for_clock([addr[-2], addr[-1]])

        self.enable_instruct = enable_insruct
        self.enable_low_byte = enable_low_byte
        self.enable_mid_byte = enable_middle_byte
        self.enable_hig_byte = enable_high_byte

        instruction_latch_clk = self.and_gate_use_for_instruct_clk([enable_insruct, pulse])
        self.inst_latch(data_out_from_memory, instruction_latch_clk)  # Code, Clock, Preset(0), Clear(0)
        Q_from_inst_latch = self.inst_latch.getQ()

        if pulse:
            print(f"Q_from_inst_latch: {Q_from_inst_latch}")

        # to halt
        Q3_from_inst_latch = Q_from_inst_latch[-4]
        self.halt_instruct = Q3_from_inst_latch
        # page 311 use invert Q3 to config flip-flip, we not use here for simple.
        # self.halt_instruct = self.invert_gate_for_halt([Q3_from_inst_latch])

        Q1_from_inst_latch = Q_from_inst_latch[-2]

        low_byte_latch_clk = self.and_gate_use_for_low_byte_clk([Q1_from_inst_latch, pulse, enable_low_byte])
        mid_byte_latch_clk = self.and_gate_use_for_middle_byte_clk([Q1_from_inst_latch, pulse, enable_middle_byte])
        high_byte_latch_clk = self.and_gate_use_for_high_byte_clk([Q1_from_inst_latch, pulse, enable_high_byte])

        Q0_from_inst_latch = Q_from_inst_latch[-1]
        outputs_from_complements_of_one = []
        for idx, xor_gate in enumerate(self.xor_gates_for_complements):
            outputs_from_complements_of_one.append(xor_gate([Q0_from_inst_latch, data_out_from_memory[idx]]))
        input_a = outputs_from_complements_of_one
        self.input_a = input_a

        if enable_low_byte:
            carry_signal_from_tri_state_buffers = 0 # low byte, no carry in    
            input_b = self.low_latch.getQ()[1:]
        elif enable_middle_byte:
            carry_signal_from_tri_state_buffers = self.low_latch.getQ()[0]  # carry out from low byte
            input_b = self.middle_latch.getQ()[1:]
        elif enable_high_byte:
            carry_signal_from_tri_state_buffers = self.middle_latch.getQ()[0]  # carry out from middle byte
            input_b = self.high_latch.getQ()
        else:
            # Default states to prevent floating wires during Instruction Fetch
            input_b = [0] * self.nbits
            carry_signal_from_tri_state_buffers = 0


        carry_in_adder = self.or_gate_for_carry_in_adder([self.and_gate_for_carry_in_adder([enable_low_byte, Q0_from_inst_latch]), carry_signal_from_tri_state_buffers])
        self.carry_in_adder = carry_in_adder
        overflow, sum_bits = self.adder(input_a, input_b, carry_in_adder)
        carry_out = self.adder.get_carry_out()

        if enable_low_byte:
            self.low_latch([carry_out]+sum_bits, low_byte_latch_clk)  # Data, Clock, Preset(0), Clear(0)
        elif enable_middle_byte:
            self.middle_latch([carry_out]+sum_bits, mid_byte_latch_clk)  # Data, Clock, Preset(0), Clear(0)
        elif enable_high_byte:
            self.high_latch(sum_bits, high_byte_latch_clk)  # Data, Clock, Preset(0), Clear(0)

        # RAM write
        Q2_from_inst_latch = Q_from_inst_latch[-3]
        ram_write = self.and_gate_for_write_ram([Q2_from_inst_latch, pulse, self.invert_for_write_ram([enable_insruct])])

        if ram_write:
            if enable_low_byte:
                self.write_to_memory(addr, self.low_latch.getQ()[1:]) # [0] is carry_out
            elif enable_middle_byte:
                self.write_to_memory(addr, self.middle_latch.getQ()[1:]) # [0] is carry_out
            elif enable_high_byte:
                self.write_to_memory(addr, self.high_latch.getQ())
            

    def write_to_memory(self, address, data):
        self.ram64KB(address, data, write=1)

    def read_low_register(self):
        return self.low_latch.getQ()[0], self.low_latch.getQ()[1:]
    
    def read_middle_register(self):
        return self.middle_latch.getQ()[0], self.middle_latch.getQ()[1:]        
    
    def read_high_register(self):
        return self.high_latch.getQ()        

In [8]:

nbits = 8
control_signals = ControlSignals()
my_oscillator = Oscillator()
my_accumulator_adder = AutomatedAccumulatingAdderV2(nbits)
my_accumulator_adder.SetMaxAddr()

# 1. write to memory 
addrs = [[0,0,0,0, 0,0,0,0, 0,0,0,0, 0,0,0,0], # Code to add next value
         [0,0,0,0, 0,0,0,0, 0,0,0,0, 0,0,0,1], # 00AFC8
         [0,0,0,0, 0,0,0,0, 0,0,0,0, 0,0,1,0],
         [0,0,0,0, 0,0,0,0, 0,0,0,0, 0,0,1,1],
        
        [0,0,0,0, 0,0,0,0, 0,0,0,0, 0,1,0,0], # Code to add next value
        [0,0,0,0, 0,0,0,0, 0,0,0,0, 0,1,0,1], # 0088B8
        [0,0,0,0, 0,0,0,0, 0,0,0,0, 0,1,1,0],
        [0,0,0,0, 0,0,0,0, 0,0,0,0, 0,1,1,1],

        [0,0,0,0, 0,0,0,0, 0,0,0,0, 1,0,0,0], # Code to store running value
        [0,0,0,0, 0,0,0,0, 0,0,0,0, 1,0,0,1],   
        [0,0,0,0, 0,0,0,0, 0,0,0,0, 1,0,1,0],   
        [0,0,0,0, 0,0,0,0, 0,0,0,0, 1,0,1,1],  
        
        [0,0,0,0, 0,0,0,0, 0,0,0,0, 1,1,0,0],  # Code to halt

        ]
datas = [2, 
         200, # C8
         175, # AF
         0, 

         2,
         184, # B8
         136, # 88
         0,

         4,
         0,
         0,
         0,

         8,
         
        ]

dst_addr1 = [0,0,0,0, 0,0,0,0, 0,0,0,0, 1,0,0,1]   
dst_addr2 = [0,0,0,0, 0,0,0,0, 0,0,0,0, 1,0,1,0]
dst_addr3 = [0,0,0,0, 0,0,0,0, 0,0,0,0, 1,0,1,1]
for addr, data in zip(addrs, datas):
    my_accumulator_adder.write_to_memory(addr,  int_to_nbit_list(data, nbits))
    # print("Initial register value:", my_accumulator_adder.ram64KB.read(addr))
    
for idx in range(50):
    if my_accumulator_adder.getHaltInst() == 1:
        print("CPU HALTED: End of program reached.")
        break # Cut the power!

    clock = my_oscillator.level()
    clk_input_to_counter, pulse = control_signals(clock)

    carry_out_low, sum_from_low = my_accumulator_adder.read_low_register() 
    carry_out_mid, sum_from_mid = my_accumulator_adder.read_middle_register()
    sum_from_hig = my_accumulator_adder.read_high_register()

    my_accumulator_adder(clk_input_to_counter, pulse)

    operand_A = my_accumulator_adder.getInputA()

    carry_in_adder = my_accumulator_adder.getCarryInAdder()
    carry_out_low_out, sum_from_low_out = my_accumulator_adder.read_low_register() 
    carry_out_mid_out, sum_from_mid_out = my_accumulator_adder.read_middle_register()
    sum_from_hig_out = my_accumulator_adder.read_high_register()

    enable_low = my_accumulator_adder.getEnableLow()
    enable_middle = my_accumulator_adder.getEnableMiddle()
    enable_high = my_accumulator_adder.getEnableHigh()



    enable_instruct = my_accumulator_adder.getEnableInstruc()

    if enable_instruct:
         print(f"FETCH CYCLE -> idx: {idx}, Clock: {clock}, Fetching Opcode at addr {addr}: {my_accumulator_adder.ram64KB.read(addr)}")
    if enable_low:
        print(f"enable_low-> idx: {idx}, Clock: {clock}, Counter Clock: {clk_input_to_counter}, Pulse: {pulse}, A: {operand_A} + B: {sum_from_low} (carry in: {carry_in_adder}) -> register: {sum_from_low_out} (carry_out: {carry_out_low_out}), dst_addr1: {my_accumulator_adder.ram64KB.read(dst_addr1)}, dst_addr2: {my_accumulator_adder.ram64KB.read(dst_addr2)}, dst_addr3: {my_accumulator_adder.ram64KB.read(dst_addr3)}")    
    if enable_middle:
        print(f"enable_middle-> idx: {idx}, Clock: {clock}, Counter Clock: {clk_input_to_counter}, Pulse: {pulse}, A: {operand_A} + B: {sum_from_mid} (carry in: {carry_in_adder}) -> register: {sum_from_mid_out} (carry_out: {carry_out_mid_out}), dst_addr1: {my_accumulator_adder.ram64KB.read(dst_addr1)}, dst_addr2: {my_accumulator_adder.ram64KB.read(dst_addr2)}, dst_addr3: {my_accumulator_adder.ram64KB.read(dst_addr3)}")    
    if enable_high:
        print(f"enable_high-> idx: {idx}, Clock: {clock}, Counter Clock: {clk_input_to_counter}, Pulse: {pulse}, A: {operand_A} + B: {sum_from_hig} (carry in: {carry_in_adder}) -> register: {sum_from_hig_out} (carry_out: {-1}), dst_addr1: {my_accumulator_adder.ram64KB.read(dst_addr1)}, dst_addr2: {my_accumulator_adder.ram64KB.read(dst_addr2)}, dst_addr3: {my_accumulator_adder.ram64KB.read(dst_addr3)}")    
        
    my_oscillator.tick()


enable_high-> idx: 0, Clock: 0, Counter Clock: 0, Pulse: 0, A: [0, 0, 0, 0, 0, 0, 0, 0] + B: [0, 0, 0, 0, 0, 0, 0, 0] (carry in: 0) -> register: [0, 0, 0, 0, 0, 0, 0, 0] (carry_out: -1), dst_addr1: [0, 0, 0, 0, 0, 0, 0, 0], dst_addr2: [0, 0, 0, 0, 0, 0, 0, 0], dst_addr3: [0, 0, 0, 0, 0, 0, 0, 0]
FETCH CYCLE -> idx: 1, Clock: 1, Fetching Opcode at addr [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0]: [0, 0, 0, 0, 1, 0, 0, 0]
FETCH CYCLE -> idx: 2, Clock: 0, Fetching Opcode at addr [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0]: [0, 0, 0, 0, 1, 0, 0, 0]
Q_from_inst_latch: [0, 0, 0, 0, 0, 0, 1, 0]
FETCH CYCLE -> idx: 3, Clock: 1, Fetching Opcode at addr [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0]: [0, 0, 0, 0, 1, 0, 0, 0]
FETCH CYCLE -> idx: 4, Clock: 0, Fetching Opcode at addr [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0]: [0, 0, 0, 0, 1, 0, 0, 0]
enable_low-> idx: 5, Clock: 1, Counter Clock: 1, Pulse: 0, A: [1, 1, 0, 0, 1, 0, 0, 0] + B: [0, 0, 0, 0, 0, 0, 0, 0] (carry in: 0) 

In [9]:

nbits = 8
control_signals = ControlSignals()
my_oscillator = Oscillator()
my_accumulator_adder = AutomatedAccumulatingAdderV2(nbits)
my_accumulator_adder.SetMaxAddr()

# 1. write to memory 
addrs = [[0,0,0,0, 0,0,0,0, 0,0,0,0, 0,0,0,0], # Code to add next value
         [0,0,0,0, 0,0,0,0, 0,0,0,0, 0,0,0,1], # 00AFC8
         [0,0,0,0, 0,0,0,0, 0,0,0,0, 0,0,1,0],
         [0,0,0,0, 0,0,0,0, 0,0,0,0, 0,0,1,1],
        
        [0,0,0,0, 0,0,0,0, 0,0,0,0, 0,1,0,0], # Code to add next value
        [0,0,0,0, 0,0,0,0, 0,0,0,0, 0,1,0,1], # 0088B8
        [0,0,0,0, 0,0,0,0, 0,0,0,0, 0,1,1,0],
        [0,0,0,0, 0,0,0,0, 0,0,0,0, 0,1,1,1],

         [0,0,0,0, 0,0,0,0, 0,0,0,0, 1,0,0,0], # Code to subtract next value
         [0,0,0,0, 0,0,0,0, 0,0,0,0, 1,0,0,1], # 00C530
         [0,0,0,0, 0,0,0,0, 0,0,0,0, 1,0,1,0],
         [0,0,0,0, 0,0,0,0, 0,0,0,0, 1,0,1,1],
        
        [0,0,0,0, 0,0,0,0, 0,0,0,0, 1,1,0,0], # Code to subtract next value
        [0,0,0,0, 0,0,0,0, 0,0,0,0, 1,1,0,1], # 00C530
        [0,0,0,0, 0,0,0,0, 0,0,0,0, 1,1,1,0],
        [0,0,0,0, 0,0,0,0, 0,0,0,0, 1,1,1,1],


        [0,0,0,0, 0,0,0,0, 0,0,0,1, 0,0,0,0], # Code to add next value
        [0,0,0,0, 0,0,0,0, 0,0,0,1, 0,0,0,1], # 030D40 
        [0,0,0,0, 0,0,0,0, 0,0,0,1, 0,0,1,0],   
        [0,0,0,0, 0,0,0,0, 0,0,0,1, 0,0,1,1],  
        
        [0,0,0,0, 0,0,0,0, 0,0,0,1, 0,1,0,0], # Code to store running total
        [0,0,0,0, 0,0,0,0, 0,0,0,1, 0,1,0,1], # 
        [0,0,0,0, 0,0,0,0, 0,0,0,1, 0,1,1,0],
        [0,0,0,0, 0,0,0,0, 0,0,0,1, 0,1,1,1],

        [0,0,0,0, 0,0,0,0, 0,0,0,1, 1,0,0,0],  # Code to halt

# Code to store running value

        ]
datas = [2, 
         200, # C8
         175, # AF
         0, 

         2,
         184, # B8
         136, # 88
         0,

        3,    # 
        80,  # 50
        195, # C3
        0,

        3,
        80, # 50
        195, # C3
        0,

        2,
        64, # 40
        13, # 0D
        3, # 03

        4,
        0,
        0,
        0,

        8,
         
        ]

dst_addr1 = [0,0,0,0, 0,0,0,0, 0,0,0,0, 1,0,0,1]   
dst_addr2 = [0,0,0,0, 0,0,0,0, 0,0,0,0, 1,0,1,0]
dst_addr3 = [0,0,0,0, 0,0,0,0, 0,0,0,0, 1,0,1,1]
for addr, data in zip(addrs, datas):
    my_accumulator_adder.write_to_memory(addr,  int_to_nbit_list(data, nbits))
    # print("Initial register value:", my_accumulator_adder.ram64KB.read(addr))
    
for idx in range(250):
    if my_accumulator_adder.getHaltInst() == 1:
        print("CPU HALTED: End of program reached.")
        break # Cut the power!

    clock = my_oscillator.level()
    clk_input_to_counter, pulse = control_signals(clock)

    carry_out_low, sum_from_low = my_accumulator_adder.read_low_register() 
    carry_out_mid, sum_from_mid = my_accumulator_adder.read_middle_register()
    sum_from_hig = my_accumulator_adder.read_high_register()

    my_accumulator_adder(clk_input_to_counter, pulse)

    operand_A = my_accumulator_adder.getInputA()

    carry_in_adder = my_accumulator_adder.getCarryInAdder()
    carry_out_low_out, sum_from_low_out = my_accumulator_adder.read_low_register() 
    carry_out_mid_out, sum_from_mid_out = my_accumulator_adder.read_middle_register()
    sum_from_hig_out = my_accumulator_adder.read_high_register()

    enable_low = my_accumulator_adder.getEnableLow()
    enable_middle = my_accumulator_adder.getEnableMiddle()
    enable_high = my_accumulator_adder.getEnableHigh()



    enable_instruct = my_accumulator_adder.getEnableInstruc()

    if enable_instruct:
         print(f"FETCH CYCLE -> idx: {idx}, Clock: {clock}, Fetching Opcode at addr {addr}: {my_accumulator_adder.ram64KB.read(addr)}")
    if enable_low:
        print(f"enable_low-> idx: {idx}, Clock: {clock}, Counter Clock: {clk_input_to_counter}, Pulse: {pulse}, A: {operand_A} + B: {sum_from_low} (carry in: {carry_in_adder}) -> register: {sum_from_low_out} (carry_out: {carry_out_low_out}), dst_addr1: {my_accumulator_adder.ram64KB.read(dst_addr1)}, dst_addr2: {my_accumulator_adder.ram64KB.read(dst_addr2)}, dst_addr3: {my_accumulator_adder.ram64KB.read(dst_addr3)}")    
    if enable_middle:
        print(f"enable_middle-> idx: {idx}, Clock: {clock}, Counter Clock: {clk_input_to_counter}, Pulse: {pulse}, A: {operand_A} + B: {sum_from_mid} (carry in: {carry_in_adder}) -> register: {sum_from_mid_out} (carry_out: {carry_out_mid_out}), dst_addr1: {my_accumulator_adder.ram64KB.read(dst_addr1)}, dst_addr2: {my_accumulator_adder.ram64KB.read(dst_addr2)}, dst_addr3: {my_accumulator_adder.ram64KB.read(dst_addr3)}")    
    if enable_high:
        print(f"enable_high-> idx: {idx}, Clock: {clock}, Counter Clock: {clk_input_to_counter}, Pulse: {pulse}, A: {operand_A} + B: {sum_from_hig} (carry in: {carry_in_adder}) -> register: {sum_from_hig_out} (carry_out: {-1}), dst_addr1: {my_accumulator_adder.ram64KB.read(dst_addr1)}, dst_addr2: {my_accumulator_adder.ram64KB.read(dst_addr2)}, dst_addr3: {my_accumulator_adder.ram64KB.read(dst_addr3)}")    
        
    my_oscillator.tick()


enable_high-> idx: 0, Clock: 0, Counter Clock: 0, Pulse: 0, A: [0, 0, 0, 0, 0, 0, 0, 0] + B: [0, 0, 0, 0, 0, 0, 0, 0] (carry in: 0) -> register: [0, 0, 0, 0, 0, 0, 0, 0] (carry_out: -1), dst_addr1: [0, 1, 0, 1, 0, 0, 0, 0], dst_addr2: [1, 1, 0, 0, 0, 0, 1, 1], dst_addr3: [0, 0, 0, 0, 0, 0, 0, 0]
FETCH CYCLE -> idx: 1, Clock: 1, Fetching Opcode at addr [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0]: [0, 0, 0, 0, 1, 0, 0, 0]
FETCH CYCLE -> idx: 2, Clock: 0, Fetching Opcode at addr [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0]: [0, 0, 0, 0, 1, 0, 0, 0]
Q_from_inst_latch: [0, 0, 0, 0, 0, 0, 1, 0]
FETCH CYCLE -> idx: 3, Clock: 1, Fetching Opcode at addr [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0]: [0, 0, 0, 0, 1, 0, 0, 0]
FETCH CYCLE -> idx: 4, Clock: 0, Fetching Opcode at addr [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0]: [0, 0, 0, 0, 1, 0, 0, 0]
enable_low-> idx: 5, Clock: 1, Counter Clock: 1, Pulse: 0, A: [1, 1, 0, 0, 1, 0, 0, 0] + B: [0, 0, 0, 0, 0, 0, 0, 0] (carry in: 0) 

In [10]:
new_dst1 = [0,0,0,0, 0,0,0,0, 0,0,0,1, 0,1,0,1] # 
new_dst2 = [0,0,0,0, 0,0,0,0, 0,0,0,1, 0,1,1,0]
new_dst3 = [0,0,0,0, 0,0,0,0, 0,0,0,1, 0,1,1,1]
print(f"new_dst1: {my_accumulator_adder.ram64KB.read(new_dst1)}, new_dst2: {my_accumulator_adder.ram64KB.read(new_dst2)}, new_dst3: {my_accumulator_adder.ram64KB.read(new_dst3)}")    

new_dst1: [0, 0, 1, 0, 0, 0, 0, 0], new_dst2: [1, 0, 1, 1, 1, 1, 1, 1], new_dst3: [0, 0, 0, 0, 0, 0, 1, 0]
